In [ ]:
import numpy as np
import tensorflow as tf
from yad2k.utils.utils import scale_boxes

In [ ]:
import argparse
import os
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import scipy.io
import scipy.misc
import pandas as pd
import PIL
from PIL import ImageFont, ImageDraw, Image

from tensorflow.python.framework.ops import EagerTensor
from tensorflow.keras.models import load_model
from yad2k.models.keras_yolo import yolo_head
from yad2k.utils.utils import draw_boxes, get_colors_for_classes, read_classes, read_anchors, preprocess_image

%matplotlib inline

In [ ]:
def yolo_filter_boxes(boxes, box_confidence, box_class_probs, threshold = .6):

    # combine "is there an object" confidence with per-class probability
    box_scores = box_class_probs * box_confidence

    # for each box, take the winning class and its score
    box_classes = tf.math.argmax(box_scores, axis=-1)
    box_class_scores = tf.math.reduce_max(box_scores, axis=-1)

    # keep only boxes whose winning score clears the threshold
    filtering_mask = box_class_scores >= threshold

    scores = tf.boolean_mask(box_class_scores, filtering_mask)
    boxes = tf.boolean_mask(boxes, filtering_mask)
    classes = tf.boolean_mask(box_classes, filtering_mask)

    return scores, boxes, classes

Thresholding on score alone still leaves plenty of overlapping boxes — the next filter, non-max suppression (NMS), handles that. *(Illustration of duplicate detections on the same car, collapsed down to one box, omitted here — image asset not included.)*

NMS leans on a function called **Intersection over Union (IoU)** — essentially, how much do two boxes overlap relative to their combined area.

**Convention used here:** origin (0,0) is the top-left of the image; x grows rightward, y grows downward. Boxes are defined by their corners, $(x_1, y_1)$ top-left and $(x_2, y_2)$ bottom-right, which makes the intersection math straightforward:

- Intersection top-left: the *larger* of the two boxes' $x_1$ / $y_1$ values
- Intersection bottom-right: the *smaller* of the two boxes' $x_2$ / $y_2$ values
- If the resulting width or height comes out negative, the boxes don't actually overlap — intersection area is 0
- Boxes that only touch at an edge or a corner also have zero overlap area

`iou()` below implements exactly that, plus the union area via $Union(A,B) = A + B - Intersection(A,B)$.

In [ ]:
def iou(box1, box2):
    """
    Arguments:
    box1 -- (box1_x1, box1_y1, box1_x2, box1_y2)
    box2 -- (box2_x1, box2_y1, box2_x2, box2_y2)
    """

    (box1_x1, box1_y1, box1_x2, box1_y2) = box1
    (box2_x1, box2_y1, box2_x2, box2_y2) = box2

    # corners of the overlapping region
    xi1 = max(box1_x1, box2_x1)
    yi1 = max(box1_y1, box2_y1)
    xi2 = min(box1_x2, box2_x2)
    yi2 = min(box1_y2, box2_y2)
    inter_width = xi2 - xi1
    inter_height = yi2 - yi1
    inter_area = max(inter_width, 0) * max(inter_height, 0)

    # union = sum of both areas minus the overlap counted twice
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union_area = box1_area + box2_area - inter_area

    iou = inter_area / union_area

    return iou

With IoU in hand, actual non-max suppression is:

1. Take the box with the highest score.
2. Compare it (via IoU) against every other box of the *same class* — drop any that overlap it past `iou_threshold`.
3. Repeat with whatever's left, until nothing above the threshold remains.

`yolo_non_max_suppression` below runs this per class (so a car overlapping a pedestrian never gets suppressed against each other), using TensorFlow's built-in `tf.image.non_max_suppression` to do the heavy lifting per class, then re-assembles and sorts the result.

In [ ]:
def yolo_non_max_suppression(scores, boxes, classes, max_boxes = 10, iou_threshold = 0.5):
    """
    Runs non-max suppression on a set of scored boxes, per class.
    """
    boxes = tf.cast(boxes, dtype=tf.float32)
    scores = tf.cast(scores, dtype=tf.float32)

    nms_indices = []
    classes_labels = tf.unique(classes)[0]  # every class that appears at least once

    for label in classes_labels:
        filtering_mask = classes == label

        # isolate just this class's boxes/scores
        boxes_label = tf.boolean_mask(boxes, filtering_mask)
        scores_label = tf.boolean_mask(scores, filtering_mask)

        if tf.shape(scores_label)[0] > 0:
            # NMS within this class only
            nms_indices_label = tf.image.non_max_suppression(
                    boxes_label,
                    scores_label,
                    max_boxes,
                    iou_threshold=iou_threshold)

            # map back to indices in the original (pre-class-split) arrays
            selected_indices = tf.squeeze(tf.where(filtering_mask), axis=1)
            nms_indices.append(tf.gather(selected_indices, nms_indices_label))

    # merge every class's surviving indices back into one list
    nms_indices = tf.concat(nms_indices, axis = 0)

    scores = tf.gather(scores, nms_indices)
    boxes = tf.gather(boxes, nms_indices)
    classes = tf.gather(classes, nms_indices)

    # keep only the top max_boxes overall, highest score first
    sort_order = tf.argsort(scores, direction='DESCENDING').numpy()
    scores = tf.gather(scores, sort_order[0:max_boxes])
    boxes = tf.gather(boxes, sort_order[0:max_boxes])
    classes = tf.gather(classes, sort_order[0:max_boxes])

    return scores, boxes, classes

In [ ]:
def yolo_boxes_to_corners(box_xy, box_wh):
    """Converts (center, width/height) box format into (corner, corner) format."""
    box_mins = box_xy - (box_wh / 2.)
    box_maxes = box_xy + (box_wh / 2.)

    return tf.keras.backend.concatenate([
        box_mins[..., 1:2],  # y_min
        box_mins[..., 0:1],  # x_min
        box_maxes[..., 1:2],  # y_max
        box_maxes[..., 0:1]  # x_max
    ])

In [ ]:
def yolo_eval(yolo_outputs, image_shape = (720, 1280), max_boxes=10, score_threshold=.6, iou_threshold=.5):
    """
    Full pipeline: takes the raw YOLO encoding and returns the final filtered
    boxes, scores, and classes.
    """

    # unpack the model's raw output
    box_xy, box_wh, box_confidence, box_class_probs = yolo_outputs

    # convert to corner format so filtering functions can use it
    boxes = yolo_boxes_to_corners(box_xy, box_wh)

    # step 1: drop low-confidence boxes
    scores, boxes, classes = yolo_filter_boxes(boxes,
                                  box_confidence,
                                  box_class_probs,
                                  score_threshold
                                 )

    # rescale from the model's 608x608 space back to the original image size
    boxes = scale_boxes(boxes, image_shape)

    # step 2: collapse overlapping duplicates via NMS
    scores, boxes, classes = yolo_non_max_suppression(scores,
                                  boxes,
                                  classes,
                                  max_boxes,
                                  iou_threshold = iou_threshold
                                 )

    return scores, boxes, classes


With the filtering pipeline done, the next step is pointing it at an actual trained YOLO model and a real image.

80 target classes, 5 anchor boxes — both are read in from text files (`coco_classes.txt`, `yolo_anchors.txt`). Source images are 720×1280 and get resized to 608×608 before going into the model.

In [ ]:
class_names = read_classes("model_data/coco_classes.txt")
anchors = read_anchors("model_data/yolo_anchors.txt")
model_image_size = (608, 608) # matches the model's input layer


Training YOLO from scratch needs a large labeled dataset and a lot of compute, so this uses pre-trained weights instead — originally from the official YOLO website, converted to a Keras-loadable format by Allan Zelener's YAD2K project (full credit in the References section). These are technically YOLOv2 weights, referred to here simply as "YOLO" for consistency with the rest of the notebook.

In [ ]:
yolo_model = load_model("model_data/", compile=False)

Layer summary of the loaded model:

In [ ]:
yolo_model.summary()

In [ ]:
def predict(image_file):
    """
    Runs the full detection pipeline on an image and displays the result.
    """

    image, image_data = preprocess_image("images/" + image_file, model_image_size = (608, 608))

    yolo_model_outputs = yolo_model(image_data)
    yolo_outputs = yolo_head(yolo_model_outputs, anchors, len(class_names))

    out_scores, out_boxes, out_classes = yolo_eval(yolo_outputs, [image.size[1],  image.size[0]], 10, 0.3, 0.5)

    print('Found {} boxes for {}'.format(len(out_boxes), "images/" + image_file))
    colors = get_colors_for_classes(len(class_names))
    draw_boxes(image, out_boxes, out_classes, class_names, out_scores)
    image.save(os.path.join("out", image_file), quality=100)
    output_image = Image.open(os.path.join("out", image_file))
    imshow(output_image)

    return out_scores, out_boxes, out_classes

Running it on `test.jpg` to check everything works end to end:

In [ ]:
out_scores, out_boxes, out_classes = predict("test.jpg")

On this image the pipeline finds 10 boxes — mostly cars, plus a bus and a traffic light, each with a confidence score and corner coordinates. Scores here range from ~0.89 down to ~0.36, which lines up with what you'd expect: closer, clearer vehicles score higher than a small traffic light in the background.

**Key takeaways**

- YOLO gets both speed and accuracy by making all predictions in a single forward pass
- The output encoding is a 19×19 grid, 5 boxes per cell, 85 numbers per box
- Score thresholding + IoU-based non-max suppression together turn hundreds of raw candidate boxes into a small, clean set of final detections
- Training YOLO from scratch is expensive (data + compute), which is why this notebook uses pre-trained weights rather than training from zero — fine-tuning on a custom dataset is a natural next step from here

This covers the full pipeline: score filtering, IoU, non-max suppression, and running a pre-trained YOLO model end-to-end on a real image. See below for everything this project builds on.

- Joseph Redmon, Santosh Divvala, Ross Girshick, Ali Farhadi — [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640) (2015)
- Joseph Redmon, Ali Farhadi — [YOLO9000: Better, Faster, Stronger](https://arxiv.org/abs/1612.08242) (2016)
- Allan Zelener — [YAD2K: Yet Another Darknet 2 Keras](https://github.com/allanzelener/YAD2K) (pre-trained weight conversion)
- Official YOLO website: https://pjreddie.com/darknet/yolo/

### Dataset

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">The Drive.ai Sample Dataset</span> is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Thanks to Brody Huval, Chih Hu, and Rahul Patel for providing this data.